In [ ]:
import pandas as pd
import numpy as np

def generate_hypothesis_dataset_v2(n_samples=1000000):
    np.random.seed(42)
    
    # Categorías principales
    groups = ['Control', 'Tratamiento']
    regions = ['Norte', 'Sur']
    
    data = {
        'user_id': np.arange(1, n_samples + 1),
        'group': np.random.choice(groups, n_samples), # A/B Test
        'region': np.random.choice(regions, n_samples),
        'age': np.random.randint(18, 65, size=n_samples)
    }
    
    df = pd.DataFrame(data)

    # --- VARIABLES PARA TEST DE MEDIAS (T-TEST) ---
    
    # 1. Caso RECHAZO: 'spent_amount' (El tratamiento gasta más)
    # Control: media 100, Tratamiento: media 105
    df['spent_amount'] = np.where(
        df['group'] == 'Control',
        np.random.normal(100, 20, n_samples),
        np.random.normal(105, 20, n_samples)
    )

    # 2. Caso NO RECHAZO: 'session_duration' (El tiempo en sitio es igual)
    # Ambos grupos: media 300 segundos
    df['session_duration'] = np.random.normal(300, 50, n_samples)

    # 3. Caso NO RECHAZO (Por Región): 'satisfaction_score' 
    # Norte y Sur tienen la misma satisfacción (media 7.5)
    df['satisfaction_score'] = np.random.normal(7.5, 1.2, n_samples)


    # --- VARIABLES PARA TEST DE PROPORCIONES (Z-TEST) ---

    # 4. Caso RECHAZO: 'converted' (El tratamiento convierte más)
    # Control: 10%, Tratamiento: 12%
    df['converted'] = np.where(
        df['group'] == 'Control',
        np.random.binomial(1, 0.10, n_samples),
        np.random.binomial(1, 0.12, n_samples)
    )

    # 5. Caso NO RECHAZO: 'click_banner' (La tasa de click es igual)
    # Ambos grupos: 5% de probabilidad
    df['click_banner'] = np.random.binomial(1, 0.05, n_samples)

    return df okdjfsdodkjf

df = generate_hypothesis_dataset_v2()

In [7]:
df.head()
df.to_csv("s9_hipotesis.csv")

In [ ]:
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

def run_hypothesis_tests(df, group_col='group'):
    """
    Ejecuta pruebas de hipótesis automáticas para variables numéricas y binarias.
    """
    results = []
    groups = df[group_col].unique()
    
    if len(groups) != 2:
        return "La función está diseñada para comparar exactamente 2 grupos (A/B)."

    group_a = df[df[group_col] == groups[0]]
    group_b = df[df[group_col] == groups[1]]

    # --- 1. TEST DE MEDIAS (T-TEST DE STUDENT) ---
    # Analizaremos variables continuas
    numeric_cols = ['spent_amount', 'session_duration', 'satisfaction_score']
    
    for col in numeric_cols:
        stat, p_value = stats.ttest_ind(group_a[col], group_b[col], equal_var=False)
        
        results.append({
            'Variable': col,
            'Tipo de Test': 'T-Test (Medias)',
            'Grupo A Mean': round(group_a[col].mean(), 2),
            'Grupo B Mean': round(group_b[col].mean(), 2),
            'P-Value': p_value,
            'Resultado': 'RECHAZAR H0' if p_value < 0.05 else 'NO RECHAZAR H0'
        })

    # --- 2. TEST DE PROPORCIONES (Z-TEST) ---
    # Analizaremos variables binarias (0 y 1)
    binary_cols = ['converted', 'click_banner']
    
    for col in binary_cols:
        count = np.array([group_a[col].sum(), group_b[col].sum()])
        nobs = np.array([len(group_a), len(group_b)])
        
        stat, p_value = proportions_ztest(count, nobs)
        
        results.append({
            'Variable': col,
            'Tipo de Test': 'Z-Test (Proporciones)',
            'Grupo A Prop': round(group_a[col].mean(), 4),
            'Grupo B Prop': round(group_b[col].mean(), 4),
            'P-Value': p_value,
            'Resultado': 'RECHAZAR H0' if p_value < 0.05 else 'NO RECHAZAR H0'
        })

    return pd.DataFrame(results)

# Ejecución
reporte_stats = run_hypothesis_tests(df)
display(reporte_stats)



,Variable,Tipo de Test,Grupo A Mean,Grupo B Mean,P-Value,Resultado,Grupo A Prop,Grupo B Prop
0,spent_amount,T-Test (Medias),100.00,105.02,0.000000e+00,RECHAZAR H0,NaN,NaN
1,session_duration,T-Test (Medias),300.01,300.03,8.568846e-01,NO RECHAZAR H0,NaN,NaN
2,satisfaction_score,T-Test (Medias),7.50,7.50,5.649461e-01,NO RECHAZAR H0,NaN,NaN
3,converted,Z-Test (Proporciones),NaN,NaN,1.201368e-223,RECHAZAR H0,0.1005,0.1205
4,click_banner,Z-Test (Proporciones),NaN,NaN,6.967723e-01,NO RECHAZAR H0,0.0498,0.0500
